In [ ]:
# 加载环境变量
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from rich import print as rprint

load_dotenv()

# 定义工具

所谓的**工具（Tool）**，本质就是一个可调用的**函数**，但是这个函数不是我们自己去调用，而是给模型调用。因此除了定义函数外，我们还需要清晰描述这个工具，让模型知道这个工具如何使用。包括下列信息：
- 工具名
- 工具的作用
- 工具需要的参数


## Pydantic Model描述参数
如果函数的参数比较多，而且比较复杂，通过pydantic model来描述参数列表。

In [ ]:
import json
from rich import print as rprint  # noqa: F811

from pydantic import BaseModel, Field
from langchain.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool


class WeatherInput(BaseModel):
    """Input for weather queries."""

    location: str = Field(description="City name or coordinates")
    include_forecast: bool = Field(default=False, description="Whether to include a 5-day forecast")


@tool(args_schema=WeatherInput)
def get_weather(location: str, include_forecast: bool = False) -> str:
    """Get current weather and optional forecast.

    Args:
        location: City name or coordinates.
        include_forecast: Whether to include a 5-day forecast.

    Returns:
        A string describing the current weather and optional forecast for the given location.
    """
    return f"Weather in {location} is sunny today, and the forecast is rainy."


rprint(json.dumps(convert_to_openai_tool(get_weather), indent=2, ensure_ascii=False))

## LLM调用工具

LLM和Agent的区别：
- LLM：模型根据用户输入的问题判断需要调用工具，但是不会主动去调用
- Agent：模型预测判断调用的工具，response中的tool_calls中会包含想要调用工具的名称和参数，Agent会根据这个信息，调用对应的工具并返回结果给大模型。大模型再将返回的结果汇总为最终结果输出。


In [ ]:
# 初始化模型
model = init_chat_model("deepseek-v4-pro")

# 模型绑定工具
model_with_tools = model.bind_tools([get_weather])

# AI 可以决定是否调用工具
# 查询天气，期望调用 get_weather
response = model_with_tools.invoke("北京未来几天的天气如何？")
rprint(response)
rprint(response.tool_calls)

# 其他问题，不调用 get_weather
response = model_with_tools.invoke("1+2=？")
rprint(response)
rprint(response.tool_calls)

## Agent 调用工具

In [ ]:
from langchain.agents import create_agent

system_prompt = """你是一个专业智能助手。
严格规则：无论用户问什么问题，你都必须先调用 web_search 搜索后再回答。即使你认为自己知道答案，也必须先搜索验证。不得跳过搜索步骤。
"""

model = init_chat_model("deepseek-v4-pro")
agent = create_agent(
    model=model,
    tools=[get_weather],
    system_prompt=system_prompt
)

In [ ]:
# 询问天气，调用 get_weather
response = agent.invoke({"messages": [HumanMessage(content="北京未来几天的天气如何？")]})
rprint(response)

# 其他问题，不调用 get_weather
response = agent.invoke({"messages": [HumanMessage(content="1+2=？")]})
rprint(response)

# 预定义Tool

LangChain中提供了很多预定义的Tool，方便我们使用。例如：
- tavily：就是一个用来做web搜索的工具

它的使用步骤是这样的：
- 注册账号，创建API_KEY
- 配置环境变量: TAVILY_API_KEY
- 安装依赖：`uv add langchain-tavily`


In [ ]:
# python invoke
import os
from tavily import TavilyClient
import json

tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))
resp = tavily_client.search("The Weeknd 2025 2026 news recent updates")
print(json.dumps(resp, indent=2, ensure_ascii=False))

In [ ]:
from langchain_core.tools import tool
from langchain_tavily import TavilySearch

search_tool = TavilySearch(
    max_results=3,
    search_depth="basic",
    include_raw_content=False,  # 禁止返回网页原始内容
    topic="news"
)


# 2. 搜索工具
@tool
def web_search(query: str) -> str:
    """通过搜索引擎执行实时网络搜索，返回与查询相关的网页摘要和链接。"""
    return search_tool.invoke({"query": query})

In [ ]:
# 创建智能体，使用预定义工具 web_search
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent

model = init_chat_model("deepseek-chat")

system_prompt = """你是一个专业智能助手。
严格规则：无论用户问什么问题，你都必须先调用 web_search 搜索后再回答。即使你认为自己知道答案，也必须先搜索验证。不得跳过搜索步骤。
"""

agent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt=system_prompt
)

In [ ]:
response = agent.invoke({"messages": [HumanMessage(content="2026世界杯战况如何？")]})
rprint(response)

# 强制调用工具

## 大模型直接调用

通过 `bind_tools` 的 `tool_choice` 参数控制模型是否以及如何调用工具。

```python
model = init_chat_model("deepseek-chat")

# 方式1: "auto"（默认）—— 模型自行决定是否调用
model.bind_tools([web_search, get_weather], tool_choice="auto")

# 方式2: "required" —— 强制必须调用，但模型自己选调哪个
model.bind_tools([web_search, get_weather], tool_choice="required")

# 方式3: "none" —— 禁止调用任何工具
model.bind_tools([web_search, get_weather], tool_choice="none")

# 方式4: 锁定指定工具 —— 只能调这一个
model.bind_tools(
    [web_search, get_weather],
    tool_choice={"type": "tool", "name": "web_search"}
)
```

## Agent调用

Agent 没有自己的 `tool_choice` 参数。想强制 Agent 调指定工具，有两种方式：
1. **底层 model 绑定 tool_choice**（硬性约束，模型每步推理都必须遵守）
2. **System Prompt 引导**（软性约束，靠语义引导）

```python
model = init_chat_model("deepseek-chat")

# 硬性约束：底层 model 绑定 tool_choice
model = model.bind_tools([web_search], tool_choice="required")

# 软性约束：System Prompt 引导
system_prompt = """
你是一个专业智能助手。
严格规则：无论用户问什么问题，你都必须先调用 web_search 搜索后再回答。
即使你认为自己知道答案，也必须先搜索验证。不得跳过搜索步骤。
"""

agent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt=system_prompt,
)
```

# Tool 实践总结

生产级 Tool 开发的五个关键要素：


1. 清晰的描述：description 是 LLM 选工具的唯一依据，写清 When to use / When NOT to use |
2. 功能单一：一个工具只做一件事，避免 `mode`/`action` 切换型参数 |
3. 三层失败防护：@retry(网络) → try-catch(业务) → Agent 纠错(语义) |
4. 返回友好字符串：返回 LLM 能直接引用的格式化文本，而非原始 JSON |
5. 可观测性：日志 + 指标，多工具 Agent 出问题时能定位到具体工具和原因 |

## 清晰的描述 + 单一功能

**清晰描述**：description 是 LLM 选工具的唯一依据。必须写清 When to use、When NOT to use、参数示例值。

**功能单一**：一个工具只做一件事。如果你发现参数里出现了 `mode=` 或 `action=` 这种"切换行为"的参数，说明该拆成多个独立工具了。

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.tools import tool


# ============================================================
# ✅ 正面案例：描述清晰 + 功能单一
# ============================================================

class StockPriceInput(BaseModel):
    """Input for stock price queries."""

    symbol: str = Field(
        description="美股股票代码，大写字母。示例: AAPL、TSLA、GOOG。不支持 A 股代码。"
    )
    include_history: bool = Field(
        default=False,
        description="是否包含近 30 天历史价格，默认仅返回当前价格。"
    )


@tool(args_schema=StockPriceInput)
def get_stock_price(symbol: str, include_history: bool = False) -> str:
    """查询美股实时股价，可选返回近 30 天历史数据。

    When to use:
        - 用户询问某美股当前价格、涨跌（如"苹果股价多少"）
        - 需要股价走势数据辅助投资决策

    When NOT to use:
        - A 股/港股查询（如"茅台今天多少钱"）——本工具不支持
        - 财报、公司基本面分析——请使用其他专用工具
        - 加密货币价格——请使用 get_crypto_price 工具

    Args:
        symbol: 美股股票代码。示例: "AAPL"、"TSLA"、"GOOG"
        include_history: 是否包含近 30 天历史数据，默认 False

    Returns:
        str: 包含当前股价、涨跌幅的文本。
    """
    return f"{symbol} 当前价格: $188.50 (+2.3%)"


# ============================================================
# ❌ 反面案例：功能不单一，靠 mode 参数切换行为
# ============================================================

@tool
def user_operation(action: str, user_id: str, data: str = "") -> str:
    """创建、更新或删除用户。action 为 create/update/delete。"""
    if action == "create":
        return f"创建用户 {user_id}"
    elif action == "update":
        return f"更新用户 {user_id}"
    elif action == "delete":
        return f"删除用户 {user_id}"
    return "未知操作"


# ============================================================
# ✅ 改进：按行为拆成三个独立工具，LLM 通过名称直接匹配
# ============================================================

@tool
def create_user(name: str, email: str) -> str:
    """创建新用户。
    When to use: 用户说"注册""新建账户""添加用户"时使用。
    """
    return f"已创建用户: {name} ({email})"


@tool
def update_user(user_id: str, name: str = "", email: str = "") -> str:
    """更新已有用户的信息。
    When to use: 用户说"修改""更新""改一下"用户信息时使用。
    """
    return f"已更新用户 {user_id} 的信息"


@tool
def delete_user(user_id: str) -> str:
    """删除指定用户。
    When to use: 用户说"删除""注销""移除"用户时使用。
    """
    return f"已删除用户 {user_id}"

## 3. 三层失败防护

一个工具调用从发起到返回，经过三层防线，每层负责不同类别的错误：

```
用户输入
  │
  ▼
┌──────────────────────────────────────────────────────┐
│  Agent（LLM）                                        │
│                                                      │
│  模型决定: "调用 web_search 搜索 2026世界杯战况"      │
│                                                      │
│  ◄── 第三层在这里：Agent 纠错                         │
│  当工具返回 "错误: ..." 字符串时，LLM 读到它，        │
│  自主决定: 换关键词重试 / 换工具 / 告知用户           │
└──────────────────┬───────────────────────────────────┘
                   │ 调用工具
                   ▼
┌──────────────────────────────────────────────────────┐
│  @tool 函数本体（web_search）                         │
│                                                      │
│  def web_search(query):                              │
│      try:                                            │
│          result = _call_api(query)  ─────────┐       │
│          return format(result)               │       │
│      except TimeoutError:    ◄── 第二层在这里 │       │
│          return "错误: 超时，请稍后重试"      │       │
│      except ValueError:                      │       │
│          return "错误: 参数无效"              │       │
│                                              │       │
│  关键: catch 后 return 字符串，不 re-raise   │       │
└──────────────────────────────────────────────┼───────┘
                                               │ 调用
                                               ▼
                              ┌────────────────────────┐
                              │  _call_api(query)      │
                              │                        │
                              │  @retry(...)  ◄── 第一层在这里
                              │  def _call_api(query): │
                              │      requests.get(...) │
                              │                        │
                              │  只重试网络瞬时故障     │
                              │  (超时、连接断开、503)  │
                              └────────────────────────┘
```

**三层各司其职，错误不跨层处理：**

| 层级 | 位置 | 处理什么 | 处理方式 |
|------|------|---------|---------|
| **第一层** | 底层 API 函数 | 网络波动、瞬时超时、503 | `@retry` 自动重试，透明恢复 |
| **第二层** | `@tool` 函数体 | API 业务错误、参数无效 | `try-catch` → return 错误描述字符串 |
| **第三层** | Agent（LLM） | 读到工具返回的错误信息 | 自主决策：修正参数重试 / 换工具 / 告知用户 |

In [ ]:
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
import httpx
from langchain_core.tools import tool
from langchain.agents import create_agent


# ╔══════════════════════════════════════════════════════════════╗
# ║              第一层：底层 API 函数 —— @retry                  ║
# ║  位置: 最底层，被 @tool 函数内部调用                          ║
# ║  职责: 网络波动、瞬时超时、503 → 自动重试，透明恢复           ║
# ║  关键: 只重试网络故障，不重试业务错误（如参数无效）           ║
# ╚══════════════════════════════════════════════════════════════╝

@retry(
    stop=stop_after_attempt(3),  # 最多重试 3 次
    wait=wait_exponential(multiplier=1, min=1, max=10),  # 指数退避: 1s → 2s → 4s
    retry=retry_if_exception_type(
        (httpx.TimeoutException, httpx.ConnectError)  # ⬅ 只重试网络故障
    ),
)
def _call_search_api(query: str) -> dict:
    """【第一层】底层 API 调用，@retry 处理网络瞬时故障。

    注意：这不是 @tool，是普通函数。@retry 在这里生效。
    只有 TimeoutException 和 ConnectError 会触发重试，
    业务错误（如 400 Bad Request）不会重试，直接向上抛给第二层。
    """
    import requests
    response = requests.get(
        "https://api.search.com/v1/search",
        params={"q": query},
        timeout=5,
    )
    response.raise_for_status()  # 非 200 响应抛异常，由第二层 catch
    return response.json()


# ╔══════════════════════════════════════════════════════════════╗
# ║        第二层：@tool 函数体 —— try-catch                      ║
# ║  位置: @tool 装饰的函数内部                                   ║
# ║  职责: API 返回的业务错误 → 转为自然语言字符串返回给 LLM      ║
# ║  关键: catch 后 return 字符串，绝不 re-raise                  ║
# ╚══════════════════════════════════════════════════════════════╝

@tool
def web_search(query: str) -> str:
    """实时网络搜索，获取最新新闻和信息。

    第一层 @retry 已经处理了网络抖动，能到这里的异常都是
    "重试 3 次后仍然失败"或"业务层面拒绝"的错误。
    本层把这些异常转成 LLM 能理解的文字，让第三层 Agent 去决策。
    """
    try:
        results = _call_search_api(query)  # ← 调第一层（@retry 已生效）
        # 正常返回：格式化为 LLM 友好的文本
        lines = [f'搜索"{query}"的结果:\n']
        for i, r in enumerate(results.get("items", [])[:3], 1):
            lines.append(f"{i}. {r['title']}")
            lines.append(f"   {r['snippet'][:200]}")
        return "\n".join(lines)

    except httpx.TimeoutException:
        # 重试 3 次后仍然超时 → 告诉 LLM 稍后重试
        return (
            f"错误: 搜索「{query}」超时，API 服务繁忙。"
            f"建议: 1) 稍后重试；2) 用更简洁的搜索词"
        )
    except ValueError as e:
        # 业务层面参数无效 → 告诉 LLM 修正参数
        return f"错误: 搜索参数无效（{e}）。请更换搜索词，避免特殊字符。"
    except Exception as e:
        # 其他未知错误 → 兜底，不 re-raise
        return f"错误: 搜索失败（{type(e).__name__}），请稍后重试或更换查询方式。"


# ╔══════════════════════════════════════════════════════════════╗
# ║          第三层：Agent —— System Prompt 引导纠错             ║
# ║  位置: Agent 的 system_prompt                                ║
# ║  职责: LLM 读到工具返回的 "错误: ..." 字符串后自主决策       ║
# ║  关键: 这不是代码逻辑，是 LLM 的语义理解和推理能力           ║
# ╚══════════════════════════════════════════════════════════════╝

SYSTEM_PROMPT_WITH_ERROR_HANDLING = """你是一个专业智能助手，可以使用工具获取实时信息。

## 工具错误处理规则（第三层防护）
当工具返回的消息以 "错误:" 开头时，说明第一层(@retry)和第二层(try-catch)都已失效，
现在需要你来判断下一步动作：

1. 超时错误 → 告知用户"搜索服务繁忙，请稍后重试"，不要反复重试
2. 参数无效 → 用更简洁的关键词（去掉特殊符号、精简句子）重新调用
3. 无结果   → 换一种表述方式重新搜索，或如实告知用户未找到
4. 同一工具连续失败 2 次后 → 停止重试，直接告知用户原因

如果你认为错误是暂时的（如超时），最多重试 1 次。
不要编造工具没有返回的信息。
"""

# ─── 组装：三层防线完整示例 ───

# 创建 Agent，System Prompt 承载第三层
# agent = create_agent(
#     model=model,
#     tools=[web_search],                         # 第二层在 @tool 函数体里
#     system_prompt=SYSTEM_PROMPT_WITH_ERROR_HANDLING,  # 第三层在这里
# )
# agent.invoke({"messages": [{"role": "user", "content": "2026世界杯战况如何？"}]})

## 返回友好字符串 + 可观测性

**返回友好字符串**：返回 LLM 能直接引用和转述的格式化文本，而不是原始 JSON 或 Python 对象。

**可观测性**：日志 + 指标收集，多工具 Agent 出问题时能定位到具体工具和原因。

In [ ]:
import json
import time
import logging
from functools import wraps
from collections import defaultdict
from langchain_core.tools import tool


# ============================================================
# 4. 返回友好字符串
# ============================================================

# ❌ 差：返回原始 JSON 或 dict，LLM 需要自行解析
@tool
def search_news_bad(query: str) -> dict:
    """搜索新闻。"""
    return {
        "query": query,
        "results": [
            {"url": "https://example.com/1", "title": "...", "content": "..."}
        ],
    }


# ✅ 好：手动格式化为 LLM 可直接引用的文本
@tool
def search_news(query: str) -> str:
    """搜索最新新闻，返回带来源链接的摘要列表。

    When to use: 用户询问最新事件、新闻、实时资讯时使用。
    """
    # 模拟搜索结果
    results = [
        {
            "title": "2026世界杯决赛：巴西夺冠",
            "url": "https://example.com/wc2026",
            "content": "北京时间7月1日凌晨，2026世界杯决赛在纽约举行。巴西队以3:2击败法国队...",
        },
        {
            "title": "AI行业新突破：大模型推理能力大幅提升",
            "url": "https://example.com/ai-news",
            "content": "多家科技公司发布了新一代AI大模型，在复杂推理任务上取得了显著进步...",
        },
    ]

    # 格式化为 LLM 友好的文本
    lines = [f'搜索"{query}"的结果（共 {len(results)} 条）:\n']
    for i, r in enumerate(results, 1):
        lines.append(f"{i}. [{r['title']}]({r['url']})")
        lines.append(f"   {r['content'][:200]}")  # 截断过长内容
        lines.append("")
    return "\n".join(lines)


# 查看格式化输出
print(search_news.invoke({"query": "2026世界杯"}))
print("---")

# ============================================================
# 5. 可观测性：日志 + 指标收集
# ============================================================

# 配置日志
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("tools")


class ToolMetrics:
    """工具调用指标收集器。在 Agent 运行结束后调用 .report() 查看统计。"""

    def __init__(self):
        self.call_count: dict[str, int] = defaultdict(int)
        self.error_count: dict[str, int] = defaultdict(int)
        self.total_time: dict[str, float] = defaultdict(float)

    def record(self, tool_name: str, elapsed_ms: float, success: bool):
        self.call_count[tool_name] += 1
        self.total_time[tool_name] += elapsed_ms
        if not success:
            self.error_count[tool_name] += 1

    def report(self) -> str:
        lines = ["\n" + "=" * 60, "📊 工具调用统计", "=" * 60]
        header = f"{'工具名':<25} {'调用':>6} {'错误率':>8} {'平均耗时':>10}"
        lines.append(header)
        lines.append("-" * 60)
        for name in self.call_count:
            count = self.call_count[name]
            avg_ms = self.total_time[name] / count
            err_rate = self.error_count[name] / count * 100
            lines.append(
                f"{name:<25} {count:>6} {err_rate:>7.1f}% {avg_ms:>8.0f}ms"
            )
        lines.append("=" * 60)
        return "\n".join(lines)


# 全局指标收集器
metrics = ToolMetrics()


def trace_tool(func):
    """工具调用追踪装饰器：记录每次调用的参数、耗时和结果。

    用法：直接装饰在 @tool 函数上即可。
    """

    @wraps(func)
    def wrapper(*args, **kwargs):
        tool_name = func.__name__
        start = time.perf_counter()
        logger.info(f"[{tool_name}] 调用开始 | args={args} | kwargs={kwargs}")
        try:
            result = func(*args, **kwargs)
            elapsed_ms = (time.perf_counter() - start) * 1000
            result_preview = str(result)[:200]
            logger.info(
                f"[{tool_name}] ✅ 成功 | {elapsed_ms:.0f}ms | {result_preview}"
            )
            metrics.record(tool_name, elapsed_ms, success=True)
            return result
        except Exception as e:
            elapsed_ms = (time.perf_counter() - start) * 1000
            logger.error(
                f"[{tool_name}] ❌ 失败 | {elapsed_ms:.0f}ms | {type(e).__name__}: {e}"
            )
            metrics.record(tool_name, elapsed_ms, success=False)
            raise

    return wrapper


# ============================================================
# 使用示例
# ============================================================

@tool
@trace_tool  # 加上这一行就有日志和指标
def get_weather_with_trace(location: str) -> str:
    """查询指定城市的当前天气。"""
    time.sleep(0.1)  # 模拟 API 调用
    return f"{location}: ☀️ 晴，25°C"


@tool
@trace_tool
def search_with_trace(query: str) -> str:
    """实时网络搜索。"""
    time.sleep(0.3)  # 模拟 API 调用
    return f'搜索"{query}"的结果: ...'


# 模拟 Agent 调用链
print(get_weather_with_trace.invoke({"location": "北京"}))
print(search_with_trace.invoke({"query": "AI新闻"}))
# 模拟一次失败
metrics.record("get_weather_with_trace", 150, success=False)

print(metrics.report())